# OpenLogic Finance - Logistic Regression Strategy Research Playbook (Boxes 1-3)

Welcome to the **Logistic Regression Strategy Research Playbook**. This notebook demonstrates the application of the **6-Box Architecture Model** to research, define, and backtest a machine-learning-driven trading strategy using Logistic Regression.

We follow the exact decoupled standard pattern:
- **Box 1 (Data Prep)**: Historical asset data ingestion using horizontal foundations.
- **Box 2 (Model Library)**: Zero-dependency mathematical signal and prediction logic, including weight projections between scaled feature space and raw feature space.
- **Box 3 (Strategy Testing)**: Full high-fidelity simulation using the QuantConnect LEAN Engine bridge.

---

## Box 1: Data Prep (`data_prep/`)

The **Data Prep** layer specializes in data ingestion, feature engineering, and feature storage. It is responsible for obtaining sanitised market datasets before passing them down to predictive libraries.

In this step, we will invoke the canonical data-prep module `fetch_asset_data` to load a **10-year historical dataset** for `SPY` (S&P 500 ETF), which has been robustly fetched from Yahoo Finance.

In [1]:
%load_ext autoreload
%autoreload 2

# 1. Setup path to import OpenLogic Finance packages
import sys
import os
import pandas as pd
import numpy as np

# Ensure root folder is in python path to allow clean imports
sys.path.append(os.path.abspath('../../'))

from data_prep.connectors.market_data.tools import fetch_asset_data

# Fetch a 10-year dataset for SPY (Box 1)
print("--- Box 1: Ingesting 10y Historical Data ---")
data_summary = fetch_asset_data(ticker="SPY", period="10y")
print("Data Summary:", data_summary)

# Load the generated CSV into a Pandas DataFrame
df = pd.read_csv(data_summary["csv_path"])
df["Date"] = pd.to_datetime(df["Date"], utc=True)
df.set_index("Date", inplace=True)

print(f"\nSuccessfully loaded {len(df)} rows of SPY daily data.")
df.head()

### Box 1 Ingestion: Agentic Explanation & Context Ingestion

Here, we demonstrate calling the Level 1 market data agent (`market_data_agent`) using the Google ADK/Antigravity paradigm to fetch and explain our market data for various expertise profiles.

In [2]:
# Import the Level 1 market data agent and fetch/explanation tools
from data_prep.connectors.market_data.agent import root_agent as market_data_agent
from data_prep.connectors.market_data.agent import fetch_and_explain

print("--- Box 1: Agent Definition & Capabilities ---")
print(f"Agent Name        : {market_data_agent.name}")
print(f"Agent Description : {market_data_agent.description}")
print("\n" + "="*60 + "\n")

print("--- Box 1 Ingestion: Beginner Explanation (Age 11+) ---")
beginner_res = fetch_and_explain(ticker="SPY", period="10y", explanation_level="beginner")
print(beginner_res["explanation"])
print("\n" + "="*60 + "\n")

print("--- Box 1 Ingestion: Academic Quantitative Explanation (Jim Simons Level) ---")
academic_res = fetch_and_explain(ticker="SPY", period="10y", explanation_level="academic")
print(academic_res["explanation"])

## Box 2: Model Library (`model_library/`)

The **Model Library** houses the mathematical, statistical, and indicator models. This layer is entirely decoupled from execution and testing frameworks; it represents the **pure, deterministic decision mathematics (the 'WHAT to decide')**.

In this quantitative playground, we will instantiate a pre-trained **Logistic Regression** model payload and showcase lightweight feature engineering, probability predictions, and weight projections.

In [3]:
from model_library.ml_zoo.logistic_regression import (
    LogisticStrategyConfig,
    LogisticModelPayload,
    engineer_features,
    predict_probability,
    project_weights
)

# 1. Initialize our strategy config and pretrained model payload
config = LogisticStrategyConfig(
    ticker="SPY",
    fast_period=50,
    slow_period=200,
    rsi_period=14,
    probability_threshold=0.5,
    position_size=1.0,
    max_drawdown_pct=0.15
)

model_payload = LogisticModelPayload(
    weights={
        "sma_ratio": 2.5,
        "rsi_norm": 0.5,
        "momentum": 1.0
    },
    intercept=0.1,
    feature_means={
        "sma_ratio": 0.005,
        "rsi_norm": 0.02,
        "momentum": 0.0003
    },
    feature_stds={
        "sma_ratio": 0.03,
        "rsi_norm": 0.35,
        "momentum": 0.015
    }
)

print("Pre-Trained Model weights (scaled feature space):")
for f, w in model_payload.weights.items():
    print(f"  - {f}: {w}")
print(f"  - intercept (bias): {model_payload.intercept}")

### Box 2 Verification: Raw Weight Projection Equivalence

One of the vital mathematical aspects of deploying standardized machine learning models in high-frequency or daily execution systems is efficiency. 

Rather than standardizing features on every tick using means and standard deviations, we can project the weights back to the **raw feature space**. Below, we demonstrate our Box 2 weight projection helper and prove that computing probabilities using raw feature space yields identical results to scaled feature space calculation.

In [4]:
raw_weights, raw_intercept = project_weights(model_payload)
print("Projected Weights (Raw Space):")
for f, w_raw in raw_weights.items():
    print(f"  - {f}: {w_raw:.4f}")
print(f"  - raw intercept: {raw_intercept:.4f}")

print("\n--- Mathematically proving raw space projection equivalence ---")

# Create sample raw market data
sample_raw = {
    "close": 105.0,
    "fast_sma": 102.0,
    "slow_sma": 100.0,
    "rsi": 60.0,
    "prev_close": 100.0,
}

# Engineer features
feats = engineer_features(sample_raw)
print(f"Engineered raw features: {feats}")

# 1. Calculate probability using scaled features method
prob_scaled = predict_probability(feats, model_payload)

# 2. Calculate probability directly using raw projected weights
z_raw = raw_intercept + sum(raw_weights[f] * feats[f] for f in feats)
prob_raw = 1.0 / (1.0 + np.exp(-z_raw))

print(f"  -> Probability (scaled method) : {prob_scaled:.8f}")
print(f"  -> Probability (raw projected) : {prob_raw:.8f}")
print(f"  -> Mathematical Equivalence   : {abs(prob_scaled - prob_raw) < 1e-12}")

## Box 3: Strategy Testing (`strategy_testing/`)

The **Strategy Testing** layer is responsible for assessing performance and estimating risk under simulation.

To test this model strategy against high-fidelity daily stock data, we interface with the QuantConnect LEAN engine using the repository's **`LeanEngineBridge`**.

We will run the bridge, which automatically syncs our Box 2 Logistic Regression module into the local QuantConnect workspace, patches parameters, and runs a premium high-fidelity cloud backtest on QuantConnect Cloud.

In [5]:
from strategy_testing.lean_engine import LeanEngineBridge

print("--- Box 3: Strategy Testing via QuantConnect LEAN Engine ---")

# Initialize the bridge adapter
bridge = LeanEngineBridge()
lean_check = bridge.check_lean_installed()
print(f"LEAN CLI Installed: {lean_check['installed']}")

if lean_check['installed']:
    print(f"LEAN CLI Version: {lean_check['version']}")
    
    # Run the backtest for SPY with our model parameters
    print("\nRunning LEAN High-Fidelity Backtest... (Syncing and pushing ML signals)")
    res = bridge.run_backtest(
        ticker='SPY',
        fast_period=50,
        slow_period=200,
        rsi_period=14,
        probability_threshold=0.5,
        max_drawdown_pct=0.99  # Disable drawdown risk stop for standard comparison
    )
    
    if res.success:
        print("\n✅ LEAN BACKTEST COMPLETED SUCCESSFULLY!")
        
        if res.full_summary:
            print("\n=== QuantConnect LEAN Backtest Results Summary ===")
            print(res.full_summary)
    else:
        print(f"\n❌ LEAN Backtest Failed: {res.stderr}")
else:
    print("\nNote: QuantConnect LEAN CLI not installed locally. Run 'pip install lean' and check environment configuration to enable high-fidelity backtests.")